# 🔁 VQA 파인튜닝 파이프라인
### Qwen2-VL-7B-Instruct + YOLO 전처리 + LoRA (A100 최적화)

**전체 흐름**
1. 패키지 설치 및 환경 설정
2. YOLO11 모델 로드 → 이미지 내 주요 객체 크롭/강조
3. Qwen2-VL-7B 4bit 로드 + LoRA 부착
4. 파인튜닝 (A100 기준 약 30~40분)
5. 추론 및 submission.csv 생성

## ① 패키지 설치

In [ ]:
# Qwen2-VL은 qwen-vl-utils 필요 (apply_chat_template 이미지 처리)
!pip -q install \
    "transformers>=4.45.0" \
    "accelerate>=0.34.2" \
    "peft>=0.13.2" \
    "bitsandbytes>=0.43.1" \
    "qwen-vl-utils>=0.0.8" \
    datasets pillow pandas torch torchvision tqdm --upgrade -q

# YOLO (ultralytics)
!pip -q install ultralytics --upgrade -q

# Pillow 버전 고정 (Qwen2-VL 이미지 처리 안정성)
!pip install --upgrade --force-reinstall Pillow==10.4.0 -q

## ② 구글 드라이브 마운트 & 데이터 압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q "/content/drive/My Drive/data.zip" -d "/content/"
!ls /content/ | head -20

## ③ 기본 설정 및 데이터 로드

In [ ]:
import os, re, math, random
import pandas as pd
import torch
from PIL import Image
from dataclasses import dataclass
from typing import Any
from torch.utils.data import Dataset, DataLoader

Image.MAX_IMAGE_PIXELS = None

# ── 디바이스 ──
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ── 주요 하이퍼파라미터 ──
MODEL_ID       = "Qwen/Qwen2-VL-7B-Instruct"   # ← 핵심 변경
IMAGE_SIZE     = 448            # 입력 해상도
MAX_NEW_TOKENS = 8
SEED           = 42

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("amp_dtype:", amp_dtype)

# ── 데이터 로드 ──
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")
print(f"train: {len(train_df)}, test: {len(test_df)}")
train_df.head(3)

## ④ YOLO 전처리 모듈

> 이미지를 VQA 모델에 넣기 전에 **YOLO11**로 주요 객체를 탐지하고,
> 검출된 박스 영역을 기준으로 **크롭+패딩**하여 더 깔끔한 입력을 전달합니다.
>
> - 객체가 1개 이상 검출되면: 가장 신뢰도 높은 박스 영역을 여유 마진과 함께 크롭
> - 객체가 없으면: 원본 이미지 그대로 사용 (안전 fallback)

In [ ]:
from ultralytics import YOLO
import numpy as np

# YOLO11n (가장 가벼운 버전, 추론 빠름)
# 더 정확도가 필요하면 yolo11m.pt 또는 yolo11l.pt 사용 가능
yolo_model = YOLO("yolo11n.pt")   # 첫 실행 시 자동 다운로드
yolo_model.to(device)

# YOLO COCO 재활용 관련 클래스 ID (병, 컵, 가위, 가방, 박스 등)
# 0=person, 39=bottle, 41=cup, 76=scissors, 24=backpack, 56=chair ...
# 재활용품과 관련 가능성 높은 클래스 화이트리스트
RECYCLE_CLASSES = {
    39,   # bottle
    41,   # cup
    44,   # bottle (wine)
    45,   # cup (wine)
    46,   # fork
    47,   # knife
    73,   # book
    74,   # clock
    76,   # scissors
    77,   # teddy bear
    79,   # toothbrush
    67,   # cell phone
    # 클래스 ID가 없어도 전체 후보 다 허용 (아래 함수 참고)
}

def yolo_crop(image: Image.Image,
              margin_ratio: float = 0.15,
              min_conf: float = 0.25,
              prefer_recycle: bool = True) -> Image.Image:
    """
    YOLO로 이미지 내 주요 객체를 탐지하고,
    신뢰도 높은 박스를 기준으로 크롭한 이미지를 반환합니다.

    Args:
        image          : PIL Image (RGB)
        margin_ratio   : 크롭 박스 주변 여유 비율 (0.15 = 15%)
        min_conf       : 최소 신뢰도 임계값
        prefer_recycle : 재활용 관련 클래스 우선 선택 여부

    Returns:
        크롭된 PIL Image (탐지 실패 시 원본 반환)
    """
    W, H = image.size

    results = yolo_model.predict(
        source=image,
        conf=min_conf,
        verbose=False,
        device=device,
    )

    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        return image  # fallback: 원본 그대로

    boxes = results[0].boxes
    xyxy   = boxes.xyxy.cpu().numpy()    # (N, 4) [x1, y1, x2, y2]
    confs  = boxes.conf.cpu().numpy()    # (N,)
    clsids = boxes.cls.cpu().numpy().astype(int)  # (N,)

    # 재활용 클래스 우선 필터 → 없으면 전체 후보 사용
    if prefer_recycle:
        mask = np.array([c in RECYCLE_CLASSES for c in clsids])
        if mask.any():
            xyxy  = xyxy[mask]
            confs = confs[mask]

    # 신뢰도 가장 높은 박스 선택
    best_idx = int(np.argmax(confs))
    x1, y1, x2, y2 = xyxy[best_idx]

    # 마진 추가
    bw = x2 - x1
    bh = y2 - y1
    x1 = max(0, x1 - bw * margin_ratio)
    y1 = max(0, y1 - bh * margin_ratio)
    x2 = min(W, x2 + bw * margin_ratio)
    y2 = min(H, y2 + bh * margin_ratio)

    # 크롭된 영역이 너무 작으면 원본 사용
    if (x2 - x1) < W * 0.05 or (y2 - y1) < H * 0.05:
        return image

    return image.crop((int(x1), int(y1), int(x2), int(y2)))


def load_and_preprocess(path: str) -> Image.Image:
    """
    이미지 로드 → YOLO 크롭 → RGB 변환
    VQA 데이터셋에서 이미지를 불러올 때 이 함수를 사용합니다.
    """
    img = Image.open(path).convert("RGB")
    img = yolo_crop(img)
    return img


# ── 빠른 시각적 테스트 ──
import matplotlib.pyplot as plt

sample_path = train_df.iloc[0]["path"]
orig = Image.open(sample_path).convert("RGB")
cropped = yolo_crop(orig)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig);     axes[0].set_title(f"원본 {orig.size}")
axes[1].imshow(cropped);  axes[1].set_title(f"YOLO 크롭 {cropped.size}")
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## ⑤ Qwen2-VL-7B 모델 로드 (4bit LoRA)

In [ ]:
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── 4bit 양자화 설정 ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=amp_dtype,
)

# ── 프로세서 ──
# Qwen2-VL은 min/max_pixels 파라미터로 입력 해상도 제어
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# ── 모델 로드 ──
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=amp_dtype,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# ── LoRA 설정 (A100 기준, 데이터 ~5000개 가정) ──
lora_config = LoraConfig(
    r=16,                               # 랭크: 표현력 ↔ 메모리 트레이드오프
    lora_alpha=32,                      # 스케일링 계수 (α/r = 2)
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj",   # Attention QKV
        "o_proj",                       # Attention Output
        "gate_proj", "up_proj", "down_proj",  # FFN
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("✅ Qwen2-VL-7B 4bit + LoRA 준비 완료")

## ⑥ 프롬프트 정의
> `hello.md`의 상세 판단 규칙 기반 프롬프트를 그대로 활용합니다.

In [ ]:
# ── 시스템 지시문 (hello.md SYSTEM_INSTRUCT) ──
SYSTEM_INSTRUCT = (
    "당신은 재활용품 이미지 기반 객관식 VQA 어시스턴트입니다. "
    "질문과 보기를 먼저 읽고, 이미지에서 확인해야 할 핵심 객체와 속성을 찾으세요. "
    "반드시 이미지와 보기의 일치 여부를 비교하여 가장 적절한 선택지를 고르세요. "
    "질문에 이미 답이 직접 포함된 경우에는 이미지보다 질문의 의미를 우선 해석하세요. "
    "출력은 반드시 a, b, c, d 중 하나의 소문자 한 글자만 하세요. "
    "설명, 공백, 문장, 부호는 절대 출력하지 마세요."
)

# ── 사용자 프롬프트 빌더 (hello.md build_mc_prompt) ──
def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        "다음은 재활용품 이미지에 대한 객관식 문제입니다.\n"
        "먼저 질문과 보기를 읽고, 사진에서 찾아야 할 대상 객체와 판단 기준을 정하세요.\n\n"

        f"질문: {question}\n\n"

        "보기:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"

        "판단 규칙:\n"
        "1. 먼저 문제와 보기를 보고 사진에서 찾아야 할 핵심 객체를 정하세요.\n"
        "2. 객체 개수를 묻는 문제라면, 동일한 객체가 2개 이상 있는지 세고, 정확하지 않으면 보기 중 가장 가까운 개수를 고르세요.\n"
        "3. 객체 종류를 묻는 문제라면, 보기 후보들을 이미지와 대조하여 사진 묘사와 다른 보기는 제외하세요.\n"
        "4. 정답 후보가 2개 이상으로 보이면, 해당 객체들의 개수를 비교하여 더 많이 보이는 객체 쪽을 우선 선택하세요.\n"
        "5. 사진을 찍은 사람의 시점에서, 가리키는 대상이나 중심 대상이 무엇인지 추정하세요.\n"
        "6. 컵과 뚜껑은 반드시 구분하세요. 컵인지, 뚜껑인지, 컵의 일부인지 주의해서 판단하세요.\n"
        "7. 플라스틱 컵이 여러 개 겹쳐 있으면, 적층된 정도를 보고 컵 개수를 추정하세요.\n"
        "8. 사진 속 물건에 글자가 보이면, 그 글자를 단서로 물건의 종류와 재질을 판단하세요.\n"
        "9. 물건의 재질은 금속(캔), 유리, 플라스틱, 종이(골판지 포함) 중 무엇에 가까운지 판단하세요.\n"
        "10. 컵라면 용기처럼 질문 자체에 답의 단서가 있는 경우에는 이미지보다 질문의 의미를 우선 반영하세요.\n"
        "11. 사진을 보고 대상의 색깔, 형태, 로고, 뚜껑 여부, 라벨 여부 등을 내부적으로 묘사한 뒤 보기와 비교하세요.\n"
        "12. 사진 묘사와 명확히 다른 보기는 정답 후보에서 제외하세요.\n"
        "13. 재질이 불확실하면 플라스틱을 우선 고려하세요. 스티로폼도 플라스틱 범주로 간주하세요.\n"
        "14. 최종적으로 가장 적절한 보기 하나를 선택하세요.\n\n"

        "출력 규칙:\n"
        "- 반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요.\n"
        "- 설명하지 마세요.\n"
        "- 다른 글자나 문장을 절대 출력하지 마세요.\n\n"

        "정답:"
    )


# ── 응답 파싱 ──
def extract_choice(text: str) -> str:
    """모델 응답에서 a/b/c/d 한 글자만 추출"""
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    # 토큰 단위 스캔
    for tok in last.split():
        if tok in ["a", "b", "c", "d"]:
            return tok
    # 전체 텍스트 스캔
    for ch in ["a", "b", "c", "d"]:
        if ch in text:
            return ch
    return "a"  # fallback

print("✅ 프롬프트 함수 정의 완료")
print("\n[테스트 프롬프트 미리보기]")
print(build_mc_prompt("사진 속 용기의 재질은 무엇인가요?", "유리", "금속", "플라스틱", "종이")[:300], "...")

## ⑦ Dataset & DataCollator

In [ ]:
class VQAMCDataset(Dataset):
    """
    VQA 객관식 데이터셋.
    train=True  → 정답 포함 (파인튜닝용)
    train=False → 정답 미포함 (추론용)
    use_yolo    → YOLO 전처리 적용 여부
    """
    def __init__(self, df, processor, train: bool = True, use_yolo: bool = True):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.train     = train
        self.use_yolo  = use_yolo

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        # ── 이미지 로드 (YOLO 전처리 포함) ──
        if self.use_yolo:
            img = load_and_preprocess(row["path"])
        else:
            img = Image.open(row["path"]).convert("RGB")

        # ── 프롬프트 구성 ──
        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role": "system",  "content": [{"type": "text",  "text": SYSTEM_INSTRUCT}]},
            {"role": "user",    "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text},
            ]},
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc

print("✅ Dataset / Collator 정의 완료")

## ⑧ 학습 하이퍼파라미터 & DataLoader

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# ── 학습 데이터 서브샘플 (A100 기준 전체 학습도 가능하나 안전하게 600개로 설정) ──
# 전체 데이터 학습 원하면 아래 줄 주석 처리
TRAIN_SAMPLE = min(600, len(train_df))
train_df_sampled = train_df.sample(n=TRAIN_SAMPLE, random_state=SEED).reset_index(drop=True)

split       = int(len(train_df_sampled) * 0.9)
train_sub   = train_df_sampled.iloc[:split]
valid_sub   = train_df_sampled.iloc[split:]

# ── 하이퍼파라미터 ──
# A100 40GB 기준: batch 2 + accum 4 = 유효 배치 8
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
EPOCHS       = 2
MAX_STEPS    = 300          # 약 40분 내 완료 (A100 기준)
LR           = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 20
MAX_NORM     = 1.0
USE_YOLO     = True         # YOLO 전처리 켜기/끄기

# ── DataLoader ──
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

train_ds = VQAMCDataset(train_sub, processor, train=True,  use_yolo=USE_YOLO)
valid_ds = VQAMCDataset(valid_sub, processor, train=True,  use_yolo=USE_YOLO)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=DataCollator(processor, train=True),
    pin_memory=True,
    num_workers=2,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=DataCollator(processor, train=True),
    pin_memory=True,
    num_workers=2,
)

# ── Optimizer / Scheduler ──
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=min(total_steps, MAX_STEPS),
)

print(f"train_batches={len(train_loader)}, valid_batches={len(valid_loader)}")
print(f"train_samples={len(train_sub)}, valid_samples={len(valid_sub)}")
print(f"YOLO 전처리: {'ON' if USE_YOLO else 'OFF'}")

## ⑨ 파인튜닝 루프

In [ ]:
import math
from tqdm.auto import tqdm

use_fp16_scaler = (torch.cuda.is_available() and amp_dtype == torch.float16)
scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)

model.train()
global_step = 0
best_val_loss = float("inf")
SAVE_DIR = "/content/qwen2_vl_7b_lora"

print(f"🚀 파인튜닝 시작 | amp={amp_dtype} | epochs={EPOCHS} | max_steps={MAX_STEPS}")

for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch} [train]", unit="batch")
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress, start=1):
        if global_step >= MAX_STEPS:
            break

        batch = {
            k: (v.to(model.device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()
        }

        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss
            if GRAD_ACCUM > 1:
                loss = loss / GRAD_ACCUM

        if use_fp16_scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % GRAD_ACCUM == 0:
            if use_fp16_scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_NORM)

            if use_fp16_scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            progress.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM:.4f}", "gs": global_step})

        running_loss += float(loss)

    if global_step >= MAX_STEPS:
        print(f"⏹️ MAX_STEPS={MAX_STEPS} 도달. 학습 종료.")
        break

    # ── Validation ──
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch} [valid]", unit="batch"):
            vb = {
                k: (v.to(model.device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
                for k, v in vb.items()
            }
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
                out = model(**vb)
                vloss = out.loss if hasattr(out, "loss") and out.loss is not None else 0.0
            val_loss += float(vloss)
            val_steps += 1

    mean_vloss = val_loss / max(val_steps, 1)
    print(f"[Epoch {epoch}] val_loss={mean_vloss:.4f}")

    # 베스트 모델 저장
    if mean_vloss < best_val_loss:
        best_val_loss = mean_vloss
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  ✅ 베스트 모델 저장 → {SAVE_DIR} (val_loss={mean_vloss:.4f})")

    model.train()

# 최종 저장 (베스트 아닐 수 있으나 안전망)
model.save_pretrained(SAVE_DIR + "_final")
processor.save_pretrained(SAVE_DIR + "_final")
print(f"\n✅ 파인튜닝 완료. 저장: {SAVE_DIR}_final")

## ⑩ 추론 (Test 데이터 → submission.csv)

In [ ]:
from tqdm.auto import tqdm

model.eval()
preds = []

print(f"🔍 추론 시작 | test_df={len(test_df)}건 | YOLO={'ON' if USE_YOLO else 'OFF'}")

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]

    # ── 이미지 로드 (YOLO 전처리 포함) ──
    if USE_YOLO:
        img = load_and_preprocess(row["path"])
    else:
        img = Image.open(row["path"]).convert("RGB")

    # ── 프롬프트 구성 ──
    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    )

    messages = [
        {"role": "system",  "content": [{"type": "text",  "text": SYSTEM_INSTRUCT}]},
        {"role": "user",    "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": user_text},
        ]},
    ]

    # ── 토크나이징 ──
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[text],
        images=[img],
        return_tensors="pt",
    ).to(model.device)

    # ── 생성 ──
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.0,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    pred = extract_choice(output_text)
    preds.append(pred)

# ── 제출 파일 생성 ──
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print(f"\n✅ submission.csv 저장 완료 ({len(submission)}건)")
print(submission["answer"].value_counts())
submission.head(10)

## ⑪ 구글 드라이브 저장

In [ ]:
import shutil

# submission.csv → 드라이브
drive_path = "/content/drive/My Drive/submission_qwen2vl7b.csv"
submission.to_csv(drive_path, index=False)
print(f"✅ 드라이브 저장: {drive_path}")

# LoRA 체크포인트 압축 → 드라이브
# (체크포인트 크기가 크므로 필요시 주석 해제)
# shutil.make_archive("/content/drive/My Drive/qwen2_lora", "zip", SAVE_DIR)
# print("✅ LoRA 체크포인트 드라이브 저장 완료")

---
## 🔧 (선택) Zero-shot 추론 전용 모드
> 파인튜닝 없이 **Qwen2-VL-7B 기본 모델**만으로 빠르게 추론하려면 이 셀만 실행하세요.
> A100에서 전체 test 추론은 약 10~20분 소요됩니다.

In [ ]:
# ══ 이 셀은 파인튜닝 없이 zero-shot만 원할 때 단독 실행 ══
# 위 셀들 (설치 ~ 프롬프트 정의) 실행 후 이 셀 실행

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

MODEL_ID  = "Qwen/Qwen2-VL-7B-Instruct"
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
device    = "cuda"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=amp_dtype,
)

processor_zs = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=448*448, max_pixels=448*448)
model_zs = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=amp_dtype,
    device_map="auto",
)
model_zs.eval()
print("✅ Zero-shot 모델 로드 완료")

test_df = pd.read_csv("/content/test.csv")
preds_zs = []

for i in tqdm(range(len(test_df)), desc="Zero-shot Inference"):
    row = test_df.iloc[i]
    img = load_and_preprocess(row["path"])  # YOLO 전처리 포함
    user_text = build_mc_prompt(str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"]))

    messages = [
        {"role": "system",  "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user",    "content": [{"type": "image", "image": img}, {"type": "text", "text": user_text}]},
    ]

    text = processor_zs.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor_zs(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model_zs.generate(**inputs, max_new_tokens=4, do_sample=False)

    output_text = processor_zs.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds_zs.append(extract_choice(output_text))

sub_zs = pd.DataFrame({"id": test_df["id"], "answer": preds_zs})
sub_zs.to_csv("/content/submission_zeroshot.csv", index=False)
sub_zs.to_csv("/content/drive/My Drive/submission_zeroshot.csv", index=False)
print("✅ Zero-shot submission 저장 완료")
print(sub_zs["answer"].value_counts())